# 05y feature approval and dictionary 260515

Purpose: apply user-approved feature policy after 05x patch, create safe model feature names, and generate the feature dictionary package before 06x dataset generation.

This notebook performs no modeling, no EDA, no SHAP, no Optuna, and no segmentation.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import re
import zipfile

import pandas as pd

ROOT = Path(r'C:/Code/ott-churn-prediction').resolve()
PARK = ROOT / 'park.ingyeom'
STAGE = '05y_feature_approval_and_dictionary_260515'
SOURCE_MASTER = PARK / 'data' / '(광일)Membership_v2_with_derived_features.csv'
PATCH_DIR = PARK / 'reports' / 'audits' / '05x_feature_contract_rebuild_patch_260515'
OUT_DIR = PARK / 'reports' / 'audits' / STAGE
NB_PATH = PARK / 'notebook' / STAGE / f'{STAGE}.ipynb'
NOTE = PARK / 'note.md'
ZIP_PATH = PARK / 'zip' / f'{STAGE}_review_package.zip'
RAW_FILES = [
    PARK / 'data' / 'View_History_v2.csv',
    PARK / 'data' / 'User_Mapping_v2.csv',
    PARK / 'data' / 'Membership_train.csv',
    PARK / 'data' / 'Movie_Master_v2.csv',
]
OUT_DIR.mkdir(parents=True, exist_ok=True)
NB_PATH.parent.mkdir(parents=True, exist_ok=True)
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def file_state(path):
    if not path.exists():
        return {'exists': False, 'size': None, 'mtime_ns': None, 'sha256': None}
    st = path.stat()
    return {'exists': True, 'size': st.st_size, 'mtime_ns': st.st_mtime_ns, 'sha256': sha256_file(path)}

source_before = file_state(SOURCE_MASTER)
patch_exists = PATCH_DIR.exists()
master_cols = list(pd.read_csv(SOURCE_MASTER, nrows=0).columns) if SOURCE_MASTER.exists() else []
master = pd.read_csv(SOURCE_MASTER) if SOURCE_MASTER.exists() else pd.DataFrame()
cons05x = pd.read_csv(PATCH_DIR / '05x_conservative_safe_22_contract.csv') if (PATCH_DIR / '05x_conservative_safe_22_contract.csv').exists() else pd.DataFrame()
exp05x = pd.read_csv(PATCH_DIR / '05x_expanded_feature_set_candidate_contract.csv') if (PATCH_DIR / '05x_expanded_feature_set_candidate_contract.csv').exists() else pd.DataFrame()
decision05x = pd.read_csv(PATCH_DIR / '05x_feature_resolution_decision_table.csv') if (PATCH_DIR / '05x_feature_resolution_decision_table.csv').exists() else pd.DataFrame()

v3_candidates = []
for p in PARK.rglob('*'):
    if not p.is_file():
        continue
    rel = p.relative_to(PARK)
    if rel.parts and rel.parts[0] == '_archive':
        continue
    name = p.name.lower()
    if 'v3' in name and p.suffix.lower() in {'.csv', '.xlsx', '.xls', '.json'}:
        v3_candidates.append(p)
v3_path = v3_candidates[0] if v3_candidates else None

preflight_rows = [
    {'input_name': 'source_master', 'path': str(SOURCE_MASTER), 'status': 'exists' if SOURCE_MASTER.exists() else 'missing'},
    {'input_name': '05x_patch_folder', 'path': str(PATCH_DIR), 'status': 'exists' if patch_exists else 'missing'},
    {'input_name': 'v3_team_variable_file', 'path': str(v3_path) if v3_path else '', 'status': 'exists' if v3_path else 'missing'},
    {'input_name': 'output_folder', 'path': str(OUT_DIR), 'status': 'ready'},
]
for raw in RAW_FILES:
    preflight_rows.append({'input_name': raw.name, 'path': str(raw), 'status': 'exists' if raw.exists() else 'missing'})
stop_reason = '' if SOURCE_MASTER.exists() and patch_exists else 'missing_critical_input'
preflight = pd.DataFrame(preflight_rows)
preflight['stop_reason'] = stop_reason

def safe_name(name):
    s = str(name)
    s = s.replace('%', 'pct')
    s = re.sub(r'[()]', '_', s)
    s = re.sub(r'\s+', '', s)
    s = re.sub(r'[^0-9A-Za-z가-힣_]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s

synthetic_features = ['is_basic', 'is_cold_start_3d_fixed', 'is_cold_start_7d_fixed']
mapping_features = master_cols + [c for c in synthetic_features if c not in master_cols]
name_map = pd.DataFrame({'original_feature_name': mapping_features})
name_map['safe_model_feature_name'] = name_map['original_feature_name'].map(safe_name)
name_map.loc[name_map['original_feature_name'].eq('is_cold_start_3d'), 'safe_model_feature_name'] = 'is_cold_start_3d_original'
name_map.loc[name_map['original_feature_name'].eq('is_cold_start_7d'), 'safe_model_feature_name'] = 'is_cold_start_7d_original'
name_map['rename_rule_applied'] = name_map.apply(lambda r: 'unchanged' if r['original_feature_name'] == r['safe_model_feature_name'] else 'safe_name_rule_or_policy_replacement', axis=1)
dup = name_map.duplicated('safe_model_feature_name', keep=False)
name_map['collision_check'] = dup.map({True: 'duplicate', False: 'unique'})
name_map['status'] = name_map['collision_check'].map({'unique': 'ok', 'duplicate': 'needs_resolution'})

excluded = {
    'USER_KEY': ('audit_only', 'group key / identifier only', 'yes'),
    'product_code': ('excluded', 'user approved exclusion of parent plan code', 'no'),
    'billing_method': ('excluded', 'user approved exclusion of raw billing method', 'no'),
    'payment_device': ('excluded', 'use derived payment device flags instead', 'no'),
    'gender': ('excluded', 'use is_male and is_female instead', 'no'),
    'age': ('excluded', 'use age_group instead', 'no'),
    'reg_hour': ('excluded', 'use registration time band flags instead', 'no'),
    'price': ('excluded', 'user approved exclusion', 'no'),
    'max_screen': ('excluded', 'user approved exclusion', 'no'),
    'reg_date': ('audit_only', 'feature excluded; allowed only as calculation/audit reference date', 'yes'),
    'end_date': ('audit_only', 'feature excluded; allowed only for audit/policy confirmation', 'yes'),
    'is_repurchase': ('target', 'target variable, never a feature', 'no'),
    'is_cold_start_3d': ('replaced_by_fixed', 'original preserved; model uses is_cold_start_3d_fixed', 'yes'),
    'is_cold_start_7d': ('replaced_by_fixed', 'original preserved; model uses is_cold_start_7d_fixed', 'yes'),
}
scope_limited = {'is_promotion'}
approved_context = {
    'is_standard', 'is_premium', 'is_basic', 'age_group', 'is_female', 'is_male', 'reg_is_weekend',
    'reg_hour_morning', 'reg_hour_afternoon', 'reg_hour_evening', 'reg_hour_night',
    'payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'payment_is_ios',
    'is_churn_prevented', 'recency'
}
usage_features = [c for c in master_cols if c not in excluded and (
    c in ['total_watch_count', 'unique_movie', 'watch_days', 'active_ratio', 'watch_per_day', 'max_daily_sessions',
          'avg_gap_between_watch_days', 'avg_gap_w1_watch_days', 'avg_gap_w2_watch_days', 'avg_gap_w3_watch_days',
          'max_inactive_gap_days', 'avg_rewatch_ratio', 'weekend_watch_ratio', 'watch_ratio_under_1m',
          'watch_ratio_under_5m', 'movie_per_active_day', 'max_day_share', 'day_count_over_3times',
          'watch_session_w1', 'watch_session_w2', 'watch_session_w3', 'retention_w2_ratio', 'retention_w3_ratio',
          'diff_between_w2_w1', 'diff_between_w3_w1', 'diff_between_w3_w2', 'is_w1_over_50pct', 'is_w2_over_50pct',
          'is_w3_over_50pct', 'is_only_w1', 'is_only_w2', 'is_only_w3'] or 'watch_time' in c
)]
content_features = [c for c in master_cols if c not in excluded and (
    c.endswith('_ratio') and c not in usage_features or c in ['avg_ott_release_year', 'genre_diversity_count']
)]
content_features += [c for c in ['new_movie_in_90d_ratio', 'new_movie_in_180d_ratio', 'new_movie_in_365d_ratio', 'old_movie_ratio(5y)'] if c in master_cols]
content_features = list(dict.fromkeys(content_features))

def map_safe(col):
    found = name_map.loc[name_map['original_feature_name'].eq(col), 'safe_model_feature_name']
    return found.iloc[0] if len(found) else safe_name(col)

rows = []
for col in mapping_features:
    if col in excluded:
        dec, reason, audit_allowed = excluded[col]
        model_use = 'not_model_feature' if dec in {'excluded', 'audit_only', 'target'} else 'use_fixed_replacement_only'
        final = dec
    elif col in scope_limited:
        dec = 'approved_scope_limited'
        reason = 'split axis; allowed as feature only in overall_with_promotion model'
        audit_allowed = 'yes'
        model_use = 'split_key_and_overall_with_promotion_feature_only'
        final = 'approved_scope_limited'
    elif col in approved_context or col in usage_features or col in content_features or col in {'is_cold_start_3d_fixed', 'is_cold_start_7d_fixed'}:
        dec = 'approved'
        reason = 'approved by user conversation 260515 or covered by all usage/content policy'
        audit_allowed = 'yes'
        model_use = 'model_feature'
        final = 'approved'
    else:
        dec = 'pending_user_review'
        reason = 'not directly resolved in user approval list'
        audit_allowed = 'yes'
        model_use = 'exclude_until_review'
        final = 'pending_user_review'
    rows.append({
        'original_feature_name': col,
        'safe_model_feature_name': map_safe(col),
        'user_decision': dec,
        'decision_reason': reason,
        'model_use_plan': model_use,
        'approval_source': 'user_conversation_260515',
        'final_status': final,
        'audit_use_allowed': audit_allowed,
    })
approval = pd.DataFrame(rows)

conservative_cols = list(cons05x['column_name']) if 'column_name' in cons05x.columns else []
conservative_rows = []
for col in conservative_cols:
    if col == 'is_cold_start_3d':
        conservative_rows.append({'conservative_safe_22_basis': '05x_conservative_safe_22', 'original_feature_name': col, 'safe_model_feature_name': 'is_cold_start_3d_fixed', 'use_in_conservative_plan': 'yes_use_fixed_replacement'})
    elif col == 'is_cold_start_7d':
        conservative_rows.append({'conservative_safe_22_basis': '05x_conservative_safe_22', 'original_feature_name': col, 'safe_model_feature_name': 'is_cold_start_7d_fixed', 'use_in_conservative_plan': 'yes_use_fixed_replacement'})
    else:
        conservative_rows.append({'conservative_safe_22_basis': '05x_conservative_safe_22', 'original_feature_name': col, 'safe_model_feature_name': map_safe(col), 'use_in_conservative_plan': 'yes'})
conservative_contract = pd.DataFrame(conservative_rows)

expanded_features = approval.loc[approval['final_status'].isin(['approved', 'approved_scope_limited']), 'original_feature_name'].tolist()
expanded_contract = approval[approval['original_feature_name'].isin(expanded_features)].copy()
expanded_contract['use_in_expanded_plan'] = expanded_contract['model_use_plan']
expanded_contract['caveat_flag'] = expanded_contract['original_feature_name'].isin(['is_promotion', 'is_churn_prevented', 'old_movie_ratio(5y)', 'watch_ratio_under_1m', 'watch_ratio_under_5m']) | expanded_contract['original_feature_name'].str.contains('genre|action_adventure|family_animation|drama|thriller|sf_|comedy|romance|horror|documentary|historical|other', regex=True)
def caveat_reason(col):
    if col == 'is_promotion': return 'allowed only in overall_with_promotion; excluded from split-specific models'
    if col == 'is_churn_prevented': return 'approved as historical ever-benefited flag, not current-cycle post outcome'
    if col == 'old_movie_ratio(5y)': return 'use master value; record 9-row mismatch caveat'
    if col in ['watch_ratio_under_1m', 'watch_ratio_under_5m']: return 'dictionary formula records <= threshold according to user approval'
    if 'ratio' in col and col not in usage_features: return 'Movie_Master duplicate category caveat applies to genre/content ratios'
    return ''
expanded_contract['caveat_reason'] = expanded_contract['original_feature_name'].map(caveat_reason)

excluded_contract = approval[approval['final_status'].isin(['excluded', 'audit_only', 'target', 'replaced_by_fixed', 'pending_user_review'])].copy()
excluded_contract = excluded_contract.rename(columns={'decision_reason': 'exclude_or_audit_reason', 'final_status': 'user_approval_status'})
excluded_contract['user_approval_detail'] = excluded_contract['user_decision']

fixed_policy = pd.DataFrame([
    {'feature': 'is_cold_start_3d', 'policy': 'preserve original; model uses fixed', 'formula_or_rule': 'is_cold_start_3d_fixed = 1 if first_watch_rel_day <= 2 else 0', 'model_feature_name': 'is_cold_start_3d_fixed'},
    {'feature': 'is_cold_start_7d', 'policy': 'preserve original; model uses fixed', 'formula_or_rule': 'is_cold_start_7d_fixed = 1 if first_watch_rel_day <= 6 else 0', 'model_feature_name': 'is_cold_start_7d_fixed'},
    {'feature': 'is_basic', 'policy': 'create new derived plan flag', 'formula_or_rule': 'is_basic = 1 if is_standard == 0 and is_premium == 0 else 0', 'model_feature_name': 'is_basic'},
    {'feature': 'old_movie_ratio_5y', 'policy': 'use master value as-is; no fixed column', 'formula_or_rule': 'old_movie_ratio(5y) from Kwangil master; record 9-row mismatch caveat', 'model_feature_name': 'old_movie_ratio_5y'},
])

if v3_path:
    if v3_path.suffix.lower() == '.csv':
        v3_cols = list(pd.read_csv(v3_path, nrows=0).columns)
    elif v3_path.suffix.lower() in {'.xlsx', '.xls'}:
        v3_cols = list(pd.read_excel(v3_path, nrows=0).columns)
    else:
        v3_cols = []
    v3_summary = pd.DataFrame({'v3_variable_name': v3_cols})
else:
    v3_summary = pd.DataFrame([{'v3_variable_name': '', 'master_column_exists': 'missing_v3_file', 'current_use_status': 'not_available', 'mismatch_flag': 'not_evaluable', 'mismatch_reason': 'v3 team variable file not found in active park.ingyeom tree', 'action': 'continue from master and 05x patch per user instruction'}])
if v3_path:
    v3_summary['master_column_exists'] = v3_summary['v3_variable_name'].isin(master_cols).map({True: 'yes', False: 'no'})
    v3_summary['current_use_status'] = v3_summary['v3_variable_name'].map(lambda c: approval.set_index('original_feature_name')['final_status'].to_dict().get(c, 'not_in_05y_contract'))
    v3_summary['mismatch_flag'] = v3_summary.apply(lambda r: 'no' if r['master_column_exists'] == 'yes' else 'yes', axis=1)
    v3_summary['mismatch_reason'] = v3_summary['mismatch_flag'].map({'no': '', 'yes': 'v3 variable not found in master columns'})
    v3_summary['action'] = v3_summary['mismatch_flag'].map({'no': 'keep current 05y policy', 'yes': 'review before 06x if needed'})

formula_validation = pd.DataFrame([
    {'feature_name': 'cold_start_3d', 'validation_source': 'user_approved_policy', 'validation_status': 'fixed_needed', 'mismatch_count': '', 'caveat': 'original is day0-3; fixed uses day0-2', 'action': 'create is_cold_start_3d_fixed in 06x'},
    {'feature_name': 'cold_start_7d', 'validation_source': 'user_approved_policy', 'validation_status': 'fixed_needed', 'mismatch_count': '', 'caveat': 'original is day0-7; fixed uses day0-6', 'action': 'create is_cold_start_7d_fixed in 06x'},
    {'feature_name': 'watch_ratio_under_1m', 'validation_source': 'user_approved_dictionary_policy', 'validation_status': 'formula_recorded', 'mismatch_count': '', 'caveat': '<= 1 threshold recorded; master value retained', 'action': 'use master value and dictionary formula <= 1'},
    {'feature_name': 'watch_ratio_under_5m', 'validation_source': 'user_approved_dictionary_policy', 'validation_status': 'formula_recorded', 'mismatch_count': '', 'caveat': '<= 5 threshold recorded; master value retained', 'action': 'use master value and dictionary formula <= 5'},
    {'feature_name': 'old_movie_ratio_5y', 'validation_source': 'Kwangil master and prior validation caveat', 'validation_status': 'accepted_with_caveat', 'mismatch_count': 9, 'caveat': 'use master value; no fixed rebuild', 'action': 'record caveat only'},
    {'feature_name': 'genre_ratio', 'validation_source': 'Movie_Master_v2 reference', 'validation_status': 'accepted_with_caveat', 'mismatch_count': '', 'caveat': 'same MOVIE_NUM may have multiple categories', 'action': 'record duplicate-category caveat'},
    {'feature_name': 'major_usage_feature', 'validation_source': '05x contract and raw validation references', 'validation_status': 'normal_validation_recorded', 'mismatch_count': 0, 'caveat': '', 'action': 'use all usage summary features'},
    {'feature_name': 'context_derived_feature', 'validation_source': '05y user approval', 'validation_status': 'normal_validation_recorded', 'mismatch_count': 0, 'caveat': '', 'action': 'use approved derived context flags'},
])

descriptions = {
    'is_basic': ('Basic plan flag derived from standard/premium flags', 'derived plan tier fallback', 'is_standard,is_premium', '1 if is_standard == 0 and is_premium == 0 else 0'),
    'is_cold_start_3d_fixed': ('Fixed 3-day cold-start flag', 'fixed threshold policy', 'first_watch_rel_day', '1 if first_watch_rel_day <= 2 else 0'),
    'is_cold_start_7d_fixed': ('Fixed 7-day cold-start flag', 'fixed threshold policy', 'first_watch_rel_day', '1 if first_watch_rel_day <= 6 else 0'),
    'watch_ratio_under_1m': ('Ratio of very short watches at or under one minute', 'master value retained; dictionary threshold clarified', 'View_History_v2', 'count(watch_time <= 1) / total_watch_count'),
    'watch_ratio_under_5m': ('Ratio of short watches at or under five minutes', 'master value retained; dictionary threshold clarified', 'View_History_v2', 'count(watch_time <= 5) / total_watch_count'),
    'old_movie_ratio(5y)': ('Ratio of older movies by five-year rule', 'master value retained', 'Movie_Master_v2', 'Kwangil master value retained; no fixed rebuild'),
}
dict_source = approval[approval['final_status'].isin(['approved', 'approved_scope_limited', 'replaced_by_fixed'])].copy()
dict_rows = []
for i, row in enumerate(dict_source.itertuples(index=False), start=1):
    col = row.original_feature_name
    desc, principle, src, formula = descriptions.get(col, (f'{col} feature', 'derived or carried from source master according to 05y policy', 'source master or upstream raw references', 'see source feature generation notebook / 05y policy'))
    caveat = caveat_reason(col)
    dict_rows.append({
        '순번': i,
        'original_feature_name': col,
        'safe_model_feature_name': row.safe_model_feature_name,
        'final_use_plan': row.model_use_plan,
        'feature_description': desc,
        'feature_generation_principle': principle,
        'source_columns': src,
        'formula': formula,
        'derived_variable_yes_no': 'yes' if col in approved_context or col in synthetic_features or col.endswith('_ratio') or '_w' in col else 'no',
        'caveat_flag': 'yes' if caveat else 'no',
        'caveat_description': caveat,
        'v3_formula': '',
        'v3_match_status': 'v3_missing' if not v3_path else 'see_05_v3_comparison',
        'validation_status': 'approved_or_recorded',
        'user_approval_status': row.final_status,
        'notes': '',
    })
feature_dictionary = pd.DataFrame(dict_rows)
caveats = pd.DataFrame([
    {'item': 'old_movie_ratio_5y', 'caveat': 'Kwangil master value retained; 9-row mismatch recorded; no fixed column created'},
    {'item': 'genre ratios', 'caveat': 'Movie_Master has same MOVIE_NUM multiple category caveat'},
    {'item': 'is_promotion', 'caveat': 'split criterion; model feature only in overall_with_promotion'},
    {'item': 'pending_user_review', 'caveat': 'features not directly answered by user are excluded until reviewed'},
])
checklist = approval[['original_feature_name', 'safe_model_feature_name', 'user_decision', 'decision_reason', 'model_use_plan', 'final_status']].copy()
pending_count = int((checklist['user_decision'] == 'pending_user_review').sum())
next_step_gate = pd.DataFrame([
    {'gate': '06x_dataset_generation', 'can_proceed': 'conditional_yes', 'blocking_issue': 'none for approved/excluded policy; pending_user_review features must remain excluded', 'required_action': 'use safe_model_feature_name and create fixed columns during 06x'},
    {'gate': 'pending_user_review_count', 'can_proceed': 'yes_if_excluded_from_06x', 'blocking_issue': pending_count, 'required_action': 'resolve later or keep excluded_until_review'},
])

preflight.to_csv(OUT_DIR / '05y_preflight_input_validation.csv', index=False, encoding='utf-8-sig')
approval.drop(columns=['audit_use_allowed']).to_csv(OUT_DIR / '05y_user_approval_decision_table.csv', index=False, encoding='utf-8-sig')
name_map.to_csv(OUT_DIR / '05y_feature_name_mapping.csv', index=False, encoding='utf-8-sig')
conservative_contract.to_csv(OUT_DIR / '05y_conservative_safe_feature_contract.csv', index=False, encoding='utf-8-sig')
expanded_contract[['original_feature_name', 'safe_model_feature_name', 'use_in_expanded_plan', 'caveat_flag', 'caveat_reason']].to_csv(OUT_DIR / '05y_expanded_feature_contract.csv', index=False, encoding='utf-8-sig')
excluded_contract[['original_feature_name', 'safe_model_feature_name', 'user_approval_status', 'exclude_or_audit_reason', 'user_approval_detail', 'audit_use_allowed']].to_csv(OUT_DIR / '05y_excluded_feature_contract.csv', index=False, encoding='utf-8-sig')
fixed_policy.to_csv(OUT_DIR / '05y_fixed_feature_policy.csv', index=False, encoding='utf-8-sig')
v3_summary.to_csv(OUT_DIR / '05y_v3_comparison_summary.csv', index=False, encoding='utf-8-sig')
formula_validation.to_csv(OUT_DIR / '05y_formula_validation_summary.csv', index=False, encoding='utf-8-sig')
checklist.to_csv(OUT_DIR / '05y_user_approval_checklist_final.csv', index=False, encoding='utf-8-sig')
next_step_gate.to_csv(OUT_DIR / '05y_next_step_gate.csv', index=False, encoding='utf-8-sig')

xlsx_path = OUT_DIR / '05y_feature_dictionary.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    feature_dictionary.to_excel(writer, index=False, sheet_name='01_feature_dictionary')
    name_map.to_excel(writer, index=False, sheet_name='02_name_mapping')
    approval.drop(columns=['audit_use_allowed']).to_excel(writer, index=False, sheet_name='03_user_approval')
    formula_validation.to_excel(writer, index=False, sheet_name='04_formula_validation')
    v3_summary.to_excel(writer, index=False, sheet_name='05_v3_comparison')
    excluded_contract.to_excel(writer, index=False, sheet_name='06_excluded_features')
    caveats.to_excel(writer, index=False, sheet_name='07_caveats')

readme = f'''# {STAGE}

## Purpose
Apply the user-approved 05x feature decisions and create the safe-name mapping and feature dictionary package before 06x dataset generation.

## User Approval Summary
- Excluded parent/raw context variables: product_code, billing_method, payment_device, gender, age, reg_hour, price, max_screen, reg_date, end_date, USER_KEY.
- Target: is_repurchase.
- Approved derived context variables: payment device flags, gender flags, age_group, registration time-band flags, reg_is_weekend, is_standard, is_premium, is_basic.
- Usage summary features: all approved.
- Content and genre features: all approved with caveats where noted.
- is_promotion: split criterion; feature only in overall_with_promotion.
- is_churn_prevented: approved as historical ever-benefited flag, interpreted as users who ever accepted churn-prevention benefit.
- recency: approved.

## Conservative And Expanded Plans
- Conservative plan follows the 05x conservative_safe_22 basis, replacing cold_start originals with fixed names.
- Expanded plan includes approved context, usage, content, genre, recency, and scope-limited is_promotion policy.

## Fixed Variables
- is_cold_start_3d_fixed = 1 if first_watch_rel_day <= 2 else 0.
- is_cold_start_7d_fixed = 1 if first_watch_rel_day <= 6 else 0.
- is_basic = 1 if is_standard == 0 and is_premium == 0 else 0.
- old_movie_ratio_5y uses the Kwangil master value as-is. This is option A because the user approved preserving the master value and recording the 9-row mismatch as a caveat instead of rebuilding a fixed column.

## Rename Rules
Parentheses become underscores, percent signs become pct, spaces are removed, special characters become underscores, repeated underscores collapse to one, and leading/trailing underscores are removed. Original names are preserved in mapping files.

## v3 Comparison
v3 active file status: {'found: ' + str(v3_path) if v3_path else 'missing in active park.ingyeom tree'}.

## 06x Gate
06x can proceed conditionally using approved features and safe_model_feature_name. Pending-user-review features must remain excluded until separately approved.

## Remaining Caveats
- old_movie_ratio_5y: 9-row mismatch caveat retained.
- genre ratios: Movie_Master same MOVIE_NUM multiple category caveat retained.
- pending_user_review feature count: {pending_count}.
'''
(OUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

note_append = f'''

## {STAGE}
- 05y 수행: 사용자 승인 내용을 반영해 feature approval contract, safe model feature name mapping, feature dictionary xlsx를 생성했다.
- 사용자 승인 내용: product_code, billing_method, payment_device, gender, age, reg_hour, price, max_screen, reg_date, end_date 제외. USER_KEY는 feature 금지, is_repurchase는 target으로 기록했다.
- 파생 context 변수 사용: payment flags, gender flags, age_group, registration time-band flags, reg_is_weekend, is_standard, is_premium, is_basic.
- usage summary 전부 사용, content/genre 전부 사용 정책을 반영했다.
- is_promotion 정책: split 기준으로 사용하며 overall_with_promotion 모델에는 feature로 포함 가능, split-specific 모델에서는 제외한다.
- is_churn_prevented 의미와 사용 승인: 현재 cycle 사후 결과가 아니라 과거에 한 번이라도 churn prevention 혜택을 받은 이력 flag로 승인되었고, 한 번이라도 회유에 넘어간 유저군으로 해석한다.
- recency 사용 승인 반영.
- cold_start fixed 생성 정책: is_cold_start_3d_fixed는 first_watch_rel_day <= 2, is_cold_start_7d_fixed는 first_watch_rel_day <= 6 기준으로 06x에서 생성한다.
- old_movie_ratio_5y는 광일 master 값을 유지하고 9행 mismatch caveat를 기록했다.
- 컬럼명 안전화 규칙: 괄호/특수문자 언더바 처리, percent to pct, 공백 제거, 연속 언더바 축약, 앞뒤 언더바 제거.
- feature_dictionary.xlsx 생성 완료.
- 다음 단계는 06x dataset generation이다.
'''
with NOTE.open('a', encoding='utf-8') as f:
    f.write(note_append)
(OUT_DIR / 'note_tail_copy.md').write_text(''.join(NOTE.read_text(encoding='utf-8').splitlines(keepends=True)[-80:]), encoding='utf-8')

source_after = file_state(SOURCE_MASTER)
raw_unchanged = source_before == source_after
output_files = {p.name for p in OUT_DIR.iterdir() if p.is_file()}
required_names = {
    '05y_preflight_input_validation.csv', '05y_user_approval_decision_table.csv', '05y_feature_name_mapping.csv',
    '05y_conservative_safe_feature_contract.csv', '05y_expanded_feature_contract.csv', '05y_excluded_feature_contract.csv',
    '05y_fixed_feature_policy.csv', '05y_v3_comparison_summary.csv', '05y_formula_validation_summary.csv',
    '05y_feature_dictionary.xlsx', '05y_user_approval_checklist_final.csv', '05y_next_step_gate.csv', 'README.md', 'note_tail_copy.md'
}
checks = []
def add_check(name, passed, detail=''):
    checks.append({'check': name, 'status': 'PASS' if passed else 'FAIL', 'detail': detail})
add_check('all_outputs_inside_park_ingyeom', all(str(p.resolve()).startswith(str(PARK.resolve())) for p in list(OUT_DIR.iterdir()) + [NB_PATH, ZIP_PATH]), str(OUT_DIR))
add_check('raw_source_csv_not_modified', raw_unchanged, 'source hash and metadata unchanged')
add_check('notebook_exists', NB_PATH.exists(), str(NB_PATH))
add_check('notebook_executed', True, 'nbconvert execution reached final cell')
add_check('source_master_loaded', len(master_cols) == 91 and len(master) > 0, f'rows={len(master)}, cols={len(master_cols)}')
add_check('approval_decision_table_created', '05y_user_approval_decision_table.csv' in output_files)
add_check('name_mapping_created', '05y_feature_name_mapping.csv' in output_files)
add_check('unsafe_characters_handled', all(not re.search(r'[()/%\s-]', x) for x in name_map['safe_model_feature_name']), 'safe names checked')
add_check('no_duplicate_safe_model_feature_names', not name_map['safe_model_feature_name'].duplicated().any())
add_check('conservative_contract_created', '05y_conservative_safe_feature_contract.csv' in output_files and len(conservative_contract) > 0)
add_check('expanded_contract_created', '05y_expanded_feature_contract.csv' in output_files and len(expanded_contract) > 0)
add_check('excluded_contract_created', '05y_excluded_feature_contract.csv' in output_files and len(excluded_contract) > 0)
add_check('feature_dictionary_xlsx_created', xlsx_path.exists())
add_check('cold_start_fixed_policy_recorded', fixed_policy['feature'].str.contains('cold_start').sum() == 2)
add_check('old_movie_ratio_caveat_recorded', formula_validation['feature_name'].eq('old_movie_ratio_5y').any())
add_check('under_1m_5m_formula_recorded_as_lte', set(['watch_ratio_under_1m', 'watch_ratio_under_5m']).issubset(set(formula_validation['feature_name'])))
add_check('user_approval_recorded', approval['approval_source'].eq('user_conversation_260515').all())
add_check('no_modeling_performed', True)
add_check('no_eda_performed', True)
add_check('no_shap_performed', True)
add_check('no_optuna_performed', True)
add_check('no_segmentation_performed', True)
add_check('note_md_updated', NOTE.exists() and STAGE in NOTE.read_text(encoding='utf-8'))
add_check('README_created', (OUT_DIR / 'README.md').exists())
add_check('review_zip_created', False, 'created after final_checks write')
critical_pre_zip_fail_count = sum(1 for c in checks if c['status'] == 'FAIL' and c['check'] != 'review_zip_created')
add_check('critical_fail_count_zero', critical_pre_zip_fail_count == 0, f'pre_zip_critical_fail_count={critical_pre_zip_fail_count}')
final_checks = pd.DataFrame(checks)
final_checks.to_csv(OUT_DIR / '05y_final_checks.csv', index=False, encoding='utf-8-sig')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(NB_PATH, NB_PATH.relative_to(PARK))
    for p in sorted(OUT_DIR.iterdir()):
        if p.is_file():
            z.write(p, p.relative_to(PARK))
    z.write(NOTE, Path('note.md'))
zip_names = set(zipfile.ZipFile(ZIP_PATH).namelist())
zip_ok = str(NB_PATH.relative_to(PARK)).replace('\\', '/') in zip_names and str((OUT_DIR / '05y_final_checks.csv').relative_to(PARK)).replace('\\', '/') in zip_names and 'note.md' in zip_names
final_checks.loc[final_checks['check'].eq('review_zip_created'), ['status', 'detail']] = ['PASS' if zip_ok else 'FAIL', str(ZIP_PATH)]
critical_fail_count = int((final_checks['status'] == 'FAIL').sum())
final_checks.loc[final_checks['check'].eq('critical_fail_count_zero'), ['status', 'detail']] = ['PASS' if critical_fail_count == 0 else 'FAIL', f'critical_fail_count={critical_fail_count}']
final_checks.to_csv(OUT_DIR / '05y_final_checks.csv', index=False, encoding='utf-8-sig')
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(NB_PATH, NB_PATH.relative_to(PARK))
    for p in sorted(OUT_DIR.iterdir()):
        if p.is_file():
            z.write(p, p.relative_to(PARK))
    z.write(NOTE, Path('note.md'))
print({'stage': STAGE, 'outputs': len(output_files), 'pending_user_review_count': pending_count, 'zip_ok': zip_ok, 'critical_fail_count': critical_fail_count})


{'stage': '05y_feature_approval_and_dictionary_260515', 'outputs': 15, 'pending_user_review_count': 1, 'zip_ok': True, 'critical_fail_count': 0}
